# Baselines on the Seven Cutoffs

Every numerical baseline, scored with CRPS on the seven frozen forecast origins
(4 event, 3 quiet -- see [`CUTOFFS.md`](CUTOFFS.md)) at horizons 1/2/4/8/13 weeks:
**35 scored points per model**.

The predictors come from `aieng.forecasting.methods` where one exists, plus three
local ones written for reasons documented in their modules:

| Predictor | Source | Why it is in the lineup |
|---|---|---|
| `last_value_naive` | package | the floor: zero-uncertainty random walk |
| `darts_ets` | package | random walk + honest bands (fits alpha = 1) |
| `darts_autoarima` | package | AICc-selected ARIMA, picks d=1 everywhere |
| `darts_kalman` | package | kept to show the defect below -- do not cite alone |
| `kalman_fixed` | [`kalman_fixed.py`](kalman_fixed.py) | same model with the multi-step variance bug corrected |
| `darts_lightgbm` | package | kept to show what level-features do to a tree |
| `lgbm_diff` | [`lgbm_differenced.py`](lgbm_differenced.py) | the same booster on changes, which is the fair test |
| `darts_linreg` | package | linear control; its coefficients sum to ~1 |
| `prophet_weekly` | [`prophet_baseline.py`](prophet_baseline.py) | trend + yearly seasonality hypothesis |
| `seasonal_naive_52` | [`seasonal_naive.py`](seasonal_naive.py) | pure annual-cycle control |

**The finding, up front:** every model that beats the naive does so by wrapping the
*same* point forecast (carry the last price forward) in calibrated uncertainty.
ETS drives its smoothing parameter to the boundary (alpha = 1), AutoARIMA picks d=1
at every origin, N4SID identifies |eig| = 0.999, and linear regression's lag
coefficients sum to ~1 -- four independent estimators concluding *random walk*.
The two models that hypothesise calendar structure instead (Prophet, seasonal
naive) lose to the floor by ~2x. So the bar the news-reading agent has to clear
is **calibration and event anticipation, not point accuracy**.

**Data governance:** MPOB is approved for use (2026-08-11), locally and in Coder;
the raw series must never be committed. `data/mpob/` stays gitignored -- populate
it with `uv run python scripts/fetch_mpob.py`.


---
## 1. Setup

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import pandas as pd


ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "implementations"))
warnings.filterwarnings("ignore")

from aieng.forecasting.methods import (
    DartsAutoARIMAPredictor,
    DartsExponentialSmoothingPredictor,
    DartsKalmanForecasterPredictor,
    DartsLightGBMPredictor,
    DartsLinearRegressionPredictor,
    LastValuePredictor,
)
from cpo.baselines import (
    DEFAULT_NUM_SAMPLES,
    attach_actuals,
    coverage,
    load_spec,
    mc_noise,
    mpob_service,
    per_origin,
    predictions_frame,
    run_predictor,
    skill_scores,
    summarise,
)
from cpo.data import MPOB_WEEKLY_SERIES_ID, naive_utc_now
from cpo.kalman_fixed import FixedKalmanPredictor
from cpo.lgbm_differenced import DifferencedLightGBMPredictor
from cpo.plots import plot_crps_by_horizon, plot_fan_grid, plot_predictor_comparison
from cpo.prophet_baseline import WeeklyProphetPredictor
from cpo.seasonal_naive import SeasonalNaivePredictor


spec = load_spec()
svc = mpob_service()
print(f"origins  : {[f'{d:%Y-%m-%d}' for d in spec.origins()]}")
print(
    f"horizons : {spec.task.horizons} weeks -> {len(spec.origins()) * len(spec.task.horizons)} scored points per model"
)

origins  : ['2024-02-02', '2024-05-03', '2024-08-30', '2024-11-29', '2025-02-28', '2025-06-20', '2025-11-28']
horizons : [1, 2, 4, 8, 13] weeks -> 35 scored points per model


---
## 2. Run every baseline

Hyperparameters live here, next to the results they produce (the repo convention --
see `sp500_forecasting/leaderboard.py`). `num_samples=500` was chosen by measuring
the Monte Carlo wobble: repeat runs vary by ~11 CRPS at 50 samples, ~3 at 500, and
going to 2000 buys nothing (see `cpo.baselines.DEFAULT_NUM_SAMPLES`). `lags=5`
follows the sp500 setup; for `lgbm_diff`, 12 scores the same as 5.


In [2]:
LAGS = 5

PREDICTORS = [
    LastValuePredictor(),
    DartsExponentialSmoothingPredictor(num_samples=DEFAULT_NUM_SAMPLES),
    DartsAutoARIMAPredictor(num_samples=DEFAULT_NUM_SAMPLES),
    DartsKalmanForecasterPredictor(num_samples=DEFAULT_NUM_SAMPLES),
    FixedKalmanPredictor(dim_x=1),
    DartsLightGBMPredictor(
        lags=LAGS,
        lags_past_covariates=None,  # no covariate panel is registered
        num_samples=DEFAULT_NUM_SAMPLES,
        lgbm_kwargs={"num_threads": 1, "n_jobs": 1, "verbosity": -1},
    ),
    DifferencedLightGBMPredictor(lags=LAGS, num_samples=DEFAULT_NUM_SAMPLES),
    DartsLinearRegressionPredictor(lags=LAGS, lags_past_covariates=None, num_samples=DEFAULT_NUM_SAMPLES),
    WeeklyProphetPredictor(),  # yearly seasonality ON, deliberately -- see section 6
    # Bounded-history variant -- see prophet_baseline.py module docstring for
    # why both are kept, and for a third variant (changepoint_range=0.98) that
    # was tried and rejected: it decouples trend recency from seasonality history
    # without truncating anything, fixes the worst origin here (908 -> 279 CRPS),
    # but is worse overall (445 mean CRPS, the worst of all three) because a
    # changepoint that close to the cutoff has no future data to confirm whether
    # it caught a real trend or ordinary noise -- confirmed at 2024-08-30, where
    # it extrapolates a temporary dip as a falling trend for the next 13 weeks.
    # This variant (history_years=4) is NOT a clean fix either: it repairs the
    # trend-boundary problem at the worst origin (908 -> 179 CRPS) but degrades
    # three others, including a new worst case (217 -> 1067 at 2025-02-28) --
    # 4 years is enough training data to bound the changepoint boundary
    # correctly but too few annual cycles (4) to fit yearly seasonality as
    # reliably as the 17-cycle full-history estimate.
    WeeklyProphetPredictor(history_years=4),
    SeasonalNaivePredictor(season_length=52),
]

frames = []
for predictor in PREDICTORS:
    result = run_predictor(predictor, spec, svc)
    frames.append(predictions_frame(result))
    print(f"  {result.predictor_id:22s} mean CRPS {result.mean_score:7.2f}   ({len(result.scores)} points)")

frame = attach_actuals(pd.concat(frames, ignore_index=True), svc)

  last_value_naive       mean CRPS  212.87   (35 points)


  darts_ets              mean CRPS  158.68   (35 points)


  darts_autoarima        mean CRPS  160.16   (35 points)


  darts_kalman           mean CRPS  177.67   (35 points)


  kalman_fixed_dim1      mean CRPS  162.93   (35 points)


...

11:50:28 - cmdstanpy - INFO - Chain [1] start processing


  darts_linreg           mean CRPS  176.46   (35 points)


11:50:28 - cmdstanpy - INFO - Chain [1] done processing


  prophet_weekly         mean CRPS  411.52   (35 points)


  prophet_weekly_4y      mean CRPS  426.06   (35 points)
  seasonal_naive_52      mean CRPS  400.44   (35 points)


---
## 3. Leaderboard

Mean CRPS alone is not enough to rank models -- the views below split it by
horizon (naive wins short range, structure wins long range), by cutoff kind (the
event/quiet design), and by origin (a model that wins on one shock and loses
everywhere else shows up here).


In [3]:
views = summarise(frame)
views["overall"]

,mean_crps
predictor,
darts_ets,158.68
darts_autoarima,160.16
kalman_fixed_dim1,162.93
lgbm_diff,170.73
darts_linreg,176.46
darts_kalman,177.67
last_value_naive,212.87
darts_lightgbm,232.47
seasonal_naive_52,400.44


In [4]:
views["by_horizon"].round(1)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
horizon,,,,,,,,,,,
1,99.8,89.2,93.6,116.9,91.3,92.5,119.4,107.7,429.0,309.8,463.6
2,88.4,68.6,63.7,118.4,92.2,75.4,87.4,106.8,435.9,311.2,426.2
4,106.7,91.6,85.5,94.3,121.1,104.7,108.9,93.6,410.4,312.5,411.0
8,234.4,246.6,313.3,352.3,255.0,249.2,357.8,253.7,318.6,489.7,324.4
13,271.5,297.5,332.2,480.4,322.7,292.8,390.8,291.9,463.7,707.0,377.0


In [5]:
views["by_kind"].round(1)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
kind,,,,,,,,,,,
event,197.6,205.0,243.0,322.0,212.7,202.6,289.4,226.6,499.8,540.2,418.3
quiet,110.2,96.9,90.5,113.0,128.1,110.1,110.9,96.2,293.7,273.9,376.6


In [6]:
per_origin(frame)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
origin_label,,,,,,,,,,,
2024-02-02 event,163.78,141.65,187.44,131.74,186.17,169.10,212.7,191.13,907.90,177.98,237.58
2024-05-03 quiet,118.73,86.52,85.73,89.86,132.68,107.39,93.9,151.96,401.96,264.79,264.88
2024-08-30 event,214.27,258.23,330.31,313.56,267.64,258.14,363.0,265.60,325.42,512.73,441.49
2024-11-29 event,203.11,187.57,189.14,448.31,199.72,176.37,257.3,209.71,548.17,397.87,713.53
2025-02-28 event,209.25,232.55,265.17,394.54,197.31,206.61,324.4,239.98,217.92,1072.00,280.56
2025-06-20 quiet,114.17,118.52,142.48,116.46,148.74,133.04,183.9,67.79,319.37,261.00,242.66
2025-11-28 quiet,97.80,85.73,43.38,132.82,102.98,89.86,54.9,68.94,159.90,296.03,622.38


Skill = fraction of the naive's CRPS removed. **Negative means worse than
assuming nothing ever changes.**


In [7]:
skill_scores(frame)

predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
1,0.165,0.253,0.216,0.021,0.236,0.225,0.098,-2.592,-1.594,-2.882
2,-0.012,0.216,0.272,-0.354,-0.055,0.137,-0.222,-3.986,-2.560,-3.875
4,0.021,0.159,0.215,0.134,-0.111,0.039,0.141,-2.768,-1.869,-2.773
8,0.345,0.311,0.124,0.015,0.287,0.304,0.291,0.110,-0.369,0.093
13,0.305,0.239,0.150,-0.229,0.174,0.251,0.253,-0.187,-0.809,0.035
all,0.248,0.255,0.165,-0.092,0.171,0.235,0.198,-0.933,-1.001,-0.881


---
## 4. Calibration -- the check CRPS alone cannot do

A `q10`-`q90` band claims to contain the truth 80% of the time. With 7 origins
per horizon the estimate moves in steps of 1/7 ~= 0.14, so read direction, not
digits: well below 0.80 is overconfidence, 1.000 is wider than needed. The naive
is 0.000 by construction (zero-width band); `darts_kalman`'s collapse at 8-13
weeks is the variance bug documented in [`kalman_fixed.py`](kalman_fixed.py).


In [8]:
cov = coverage(frame)
print(f"nominal coverage: {cov.attrs['nominal']:.0%}")
cov

nominal coverage: 80%


predictor,darts_autoarima,darts_ets,darts_kalman,darts_lightgbm,darts_linreg,kalman_fixed_dim1,last_value_naive,lgbm_diff,prophet_weekly,prophet_weekly_4y,seasonal_naive_52
horizon,,,,,,,,,,,
1,0.857,0.857,0.857,0.571,0.857,0.857,0.0,0.286,0.571,0.714,0.714
2,0.857,1.000,1.000,0.714,1.000,1.000,0.0,0.286,0.571,0.571,0.714
4,1.000,1.000,0.714,0.714,1.000,1.000,0.0,0.857,0.714,0.714,0.714
8,1.000,0.571,0.286,0.429,0.857,0.857,0.0,0.429,0.571,0.571,1.000
13,0.714,0.714,0.286,0.286,0.857,0.714,0.0,0.714,0.571,0.143,0.857


---
## 5. The pictures

CRPS by horizon first: the crossover between the naive and everything else is
the story of this series -- a random walk is near-unbeatable at 1-2 weeks, and
honest uncertainty pays from 4 weeks out.


In [9]:
core = frame[frame.predictor.isin(["last_value_naive", "darts_ets", "darts_autoarima", "kalman_fixed_dim1"])]
plot_crps_by_horizon(core)

The fan grid is the headline figure: one predictor across all seven cutoffs,
events on top, quiets below, one shared y-scale. Read it as *"was the model
calibrated, and where did it earn its score"* -- e.g. 2024-08-30 is the
expensive panel because the realised rally rides the upper edge of the band.


In [10]:
history = svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=naive_utc_now())
plot_fan_grid(frame, history, predictor="darts_ets")

The same grid for the naive is the contrast that explains the whole leaderboard:
an identical point forecast with no band at all.


In [11]:
plot_fan_grid(frame, history, predictor="last_value_naive")

The fan grids above are one model at a time -- good for a deep dive, hard to use for
direct comparison since each is its own figure. This overlays every predictor's median
forecast on the same axes: click a name in the legend to isolate it, click again to bring
it back, and compare any subset directly against `actual` and against each other.


In [12]:
plot_predictor_comparison(frame, history)

---
## 6. Monte Carlo noise -- how big must a gap be to mean anything?

The sampled predictors rescore differently every run. Any leaderboard gap inside
this spread is sampling noise, not evidence.


In [13]:
noise = mc_noise(
    lambda: DartsExponentialSmoothingPredictor(num_samples=DEFAULT_NUM_SAMPLES), runs=5, spec=spec, data_service=svc
)
print(f"darts_ets over {int(noise['runs'])} runs: mean {noise['mean']:.1f}, spread {noise['spread']:.1f} CRPS")
print("-> ETS vs AutoARIMA vs kalman_fixed (~157-164) is a statistical tie on 35 points.")

darts_ets over 5 runs: mean 157.7, spread 5.3 CRPS
-> ETS vs AutoARIMA vs kalman_fixed (~157-164) is a statistical tie on 35 points.


---
## 7. What this establishes, and what it does not

**Established:**

- **Every working model converges on the random walk.** ETS fits alpha = 1 at all
  seven origins, AutoARIMA selects d=1 everywhere, N4SID identifies |eig| = 0.999,
  linreg's lag coefficients sum to ~1. Their entire advantage over the naive
  (~26% CRPS) is calibrated uncertainty around the same point forecast.
- **The calendar-structure hypothesis fails.** Prophet (trend + yearly cycle) and
  the 52-week seasonal naive lose to the floor by ~2x: palm oil in this sample has
  no annual cycle worth modelling. Prophet also has no autoregressive term, so its
  1-week forecast sits ~190 RM from the last price where ARIMA sits ~40 RM.
  (Tuning its changepoints improves 411 -> ~227 but cannot fix the anchoring;
  the committed config keeps the hypothesis-test defaults.)
- **Two upstream defects were found, diagnosed, and fixed locally** --
  `darts_kalman`'s frozen multi-step variance ([`kalman_fixed.py`](kalman_fixed.py))
  and LightGBM's level-feature regime matching ([`lgbm_differenced.py`](lgbm_differenced.py)).
  Report neither package model's score without the caveat.
- **The bar for the agent:** beat ~157 CRPS not by better point forecasts (four
  estimators say there is nothing to find in the price series alone) but by
  anticipating event windows from news and staying calibrated on quiet ones.

**Limitations:**

- **35 points cannot separate close models.** ETS / AutoARIMA / kalman_fixed sit
  within ~6 CRPS of each other with ~3 CRPS of MC wobble -- a tie. The dense
  weekly backtest (`cpo_backtest.yaml`, 52 origins) is the instrument for picking
  a single champion; these seven origins are the narrative set.
- **Coverage on 7 origins is directional only** (steps of 1/7).
- **Event cutoffs were chosen with hindsight** -- valid for the controlled
  comparison, not a live forecasting record.
- **No 2022-scale shock is in the window.** The largest move any model faces here
  is ~7.7%; behaviour under a 30% shock is untested.

**Next:** the news-reading agent on the same spec, compared against `darts_ets`
on these same tables.
